In [434]:
import sys
import os
import glob
import shutil

import numpy as np
import subprocess

import ase.atoms
import ase.visualize

import arkane.encorr.reference
import arkane.encorr.corr
import arkane.ess
import arkane.encorr.bac
import arkane.exceptions
import arkane.common
import rmgpy.molecule

sys.path.append(os.environ['DFT_DIR'])
import autotst_wrapper

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun


import collections

# log to ethalpy
# from collections import defaultdict, Counter
# import os
# import re

import rdkit.Chem # import GetPeriodicTable

import rmgpy.quantity # import ScalarQuantity
import rmgpy.statmech #import HarmonicOscillator, IdealGasTranslation, LinearRotor, NonlinearRotor
import rmgpy.thermo #import ThermoData

# from arkane.common import symbol_by_number
# from arkane.encorr.corr import get_atom_correction, assign_frequency_scale_factor
# from arkane.encorr.reference import CalculatedDataEntry, ReferenceDatabase
import arkane.modelchem  # import LevelOfTheory, CompositeLevelOfTheory


In [2]:
# load the reference database
database = arkane.encorr.reference.ReferenceDatabase()
database.load()

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3);
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3);
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual v

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
ERROR:root:Unable to generate identifier for this molecule:
1 O u0 p3 c-1 {3,S}
2 O u0 p2 c0 {3,D}
3 C u0 p0 c0 {1,S} {2,D} {4,S}
4 H u0 p0 c0 {3,S}

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valen

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted

# Only collect uncharged C,H,O species

In [3]:
# Redo Single Points where geometries don't match corresponding gaussian files
special_set = []
for i in range(len(database.reference_sets['main'])):
    if database.reference_sets['main'][i].charge != 0:
        continue
    if 'N' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'S' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'CL' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'BR' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'F' in database.reference_sets['main'][i].smiles.upper():
        continue
    special_set.append(i)

# Check progress on geometry optimizations

In [4]:
def has_right_modes(cf):
    if not cf.modes:
        return False
    elif not any(isinstance(mode, rmgpy.statmech.IdealGasTranslation) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, (rmgpy.statmech.LinearRotor, rmgpy.statmech.NonlinearRotor)) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, rmgpy.statmech.HarmonicOscillator) for mode in cf.modes):
        return False
    return True

In [291]:
working_dir = '/scratch/harris.se/guassian_scratch/bac'

incomplete_geo = []
bad_geo = []
complete_geo = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    if not os.path.exists(sp_logfile):
        incomplete_geo.append(i)
        continue
    try:
        gl = arkane.ess.factory.ess_factory(sp_logfile)
    except arkane.exceptions.LogError:
        incomplete_geo.append(i)
        continue
    
    cf, freqs = gl.load_conformer()
    if not has_right_modes(cf):
        bad_geo.append(i)
    elif gl.load_force_constant_matrix() is None:
        bad_geo.append(i)
    else:
        complete_geo.append(i)

In [292]:
print(len(complete_geo))

151


In [293]:
print(len(bad_geo))

2


In [295]:
bad_geo

[301, 355]

In [296]:
print(len(incomplete_geo))

0


In [297]:
incomplete_geo

[]

In [ ]:
autotst_wrapper.check_hessian_cartesian_consistent('/scratch/harris.se/guassian_scratch/bac/species_0038/sp.log')

In [ ]:
ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][38].adjacency_list)
database_fun.get_unique_species_index(ref_sp)

In [ ]:
inconsistent

### Add check for consistent hessian ###

In [42]:
inconsistent = []
for i in complete_geo:
    ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][i].smiles)
    if len(ref_sp.molecule[0].atoms) < 14:
        continue  # the check doesn't really work on these
    
    if i in [157]:  # 157 has no rotors (cylo-pentane), so this check doesn't mean anything
        continue
    
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    print(i)
    try:
        if not autotst_wrapper.check_hessian_cartesian_consistent(sp_logfile):
            inconsistent.append(i)
            print(i, '\tinconsistent')
    except ValueError:
        inconsistent.append(i)
        print(i, '\tFailed')
        

4
5
12
15
25
33
45
46
58
62
78
83
84
87
101
110
112
153
154
214
219
220
296
343
360
373
384
402
404
406
408
409
411
412
413
414
415
416


In [43]:
# # move iop calcs to replace sp.log
# for i in inconsistent:
#     iop_logfile = os.path.join(working_dir, f'species_{i:04}', 'iop_recalc.log')
#     og_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
#     if os.path.exists(iop_logfile):
#         print(f'shifting {i}')
#         shutil.move(og_logfile, os.path.join(working_dir, f'species_{i:04}', 'sp2.log'))
#         shutil.copyfile(iop_logfile, os.path.join(working_dir, f'species_{i:04}', 'sp.log'))

In [44]:
inconsistent

[]

# Remake geometry optimization files

In [ ]:
# for i in range(len(ref_db.reference_sets['main'])):
my_indices = inconsistent
for i in my_indices:

    # get starting geometry from previous calculations
    preferred_methods = ['ccsd(t)f12', 'cbsqb32023']
    atoms = None
    for preferred_method in preferred_methods:
        for key in database.reference_sets['main'][i].calculated_data.keys():
            comparison = key
            if type(key) != arkane.modelchem.LevelOfTheory:
                comparison = key.energy
            
            if comparison.method == preferred_method:
                syms = database.reference_sets['main'][i].calculated_data[key].xyz_dict['symbols']
                xyz = database.reference_sets['main'][i].calculated_data[key].xyz_dict['coords']
                atoms = ase.Atoms(symbols=syms, positions=xyz)
                break
        if atoms:
            break
    else:
        raise ValueError(f'no preferred level of theory for entry {i}')


    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    os.makedirs(sp_dir, exist_ok=True)
    
    # write the gaussian calculation file
    with open(os.path.join(sp_dir, 'sp.com'), 'w') as f:
        ase.io.gaussian.write_gaussian_in(
            f,
            atoms,
            properties=['energy'],
            method='m062x',
            basis='cc-pvtz',
            mult=database.reference_sets['main'][i].multiplicity,
            charge=database.reference_sets['main'][i].charge,
            opt='calcfc,maxcycles=900',
            freq='',
            extra='IOP(7/33=1,2/9=2000,2/16=3)'
        )

# write the slurm script
run_script = os.path.join(working_dir, 'run.sh')
with open(run_script, 'w') as f:
    f.write("""#!/bin/bash
#SBATCH --job-name=g16_bac_opt
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem=20Gb
#SBATCH --time=24:00:00
#SBATCH --cpus-per-task=16
""" +
f'#SBATCH --array={autotst_wrapper.ordered_array_str(my_indices)}%10\n' +
"""
export GAUSS_SCRDIR=/scratch/harris.se/guassian_scratch
mkdir -p $GAUSS_SCRDIR
module load gaussian/g16
source /shared/centos7/gaussian/g16/bsd/g16.profile

RUN_i=$(printf "%04.0f" $(($SLURM_ARRAY_TASK_ID)))

cd "species_${RUN_i}"
g16 sp.com

""")


### See what's missing from the autoscience database

In [10]:
not_in_db = []
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue

In [ ]:
not_in_db = [rmgpy.species.Species(smiles='C#CC#C')]

In [ ]:
# database_fun.add_species_to_database(not_in_db)

In [ ]:
bad_geo

# Copy completed geometry optimizations from autoscience database

In [280]:
# for i in bad_geo:
skiplist = []
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue
        
    if db_index in skiplist:
        print('skipping', i)
        continue
        
    print(f'Looking for db index {db_index}')
    # see if there's a complete file
    try:
        rotor_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'conformer_*.log'))[0]
    except IndexError:
        rotor_logfile = None
        
    try:
        iop_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'iop_recalc_*.log'))[0]
    except IndexError:
        iop_logfile = None
        
    try:
        arkane_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'arkane', 'conformer_*.log'))[0]
    except IndexError:
        arkane_logfile = None
    
    my_logfile = None
    if iop_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(iop_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = iop_logfile
        except arkane.exceptions.LogError:
            print(f'Bad IOP')
            continue
    elif rotor_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(rotor_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = rotor_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Rotor')
            continue
    elif arkane_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(arkane_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = arkane_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Arkane')
            continue
    else:
        print('nothing to copy')
        continue
    
    dest_file = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    print(f'copying {my_logfile} to {dest_file}')
    shutil.copyfile(my_logfile, dest_file)
    

Looking for db index 1016
Bad Rotor
Looking for db index 1024
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_1024/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0341/sp.log
Looking for db index 1027
nothing to copy


In [315]:
species_index = 1027
conformer_dir = os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{species_index:04}', 'conformers')
rotor_dir = os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{species_index:04}', 'rotors')
conformer_file = autotst_wrapper.get_lowest_valid_conformer(conformer_dir)

# conformer_file = autotst_wrapper.get_lowest_energy_gaussian_file(conformer_dir)

os.makedirs(rotor_dir, exist_ok=True)
shutil.copyfile(conformer_file, os.path.join(rotor_dir, os.path.basename(conformer_file)))

2025-02-12 14:27:15.274535 Lowest energy conformer is /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_1027/conformers/conformer_0000.log


'/work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_1027/rotors/conformer_0000.log'

In [327]:
inconsistent

[]

In [325]:
bad_geo

[301]

In [326]:
incomplete_geo

[]

# Check progress on single-point calculations

In [412]:
incomplete_sp = []
bad_sp = []
complete_sp = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
    if i in inconsistent or i in bad_geo or i in incomplete_geo:
        continue
    
    
    orca_logfile = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    if not os.path.exists(orca_logfile):
        incomplete_sp.append(i)
        continue
    try:
        ol = arkane.ess.factory.ess_factory(orca_logfile)
        ol.load_energy()
    except arkane.exceptions.LogError:
        incomplete_sp.append(i)
        continue
        
    # make sure that the coordinates match between orca and Gaussian
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    
    g_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry()
    o_coords, o_num, o_mass = ol.load_geometry()
    
    if not np.all(np.equal(np.array(g_coords), np.array(o_coords))):
        g2_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry()
        if not np.all(np.equal(np.array(g2_coords), np.array(o_coords))):
            bad_sp.append(i)
            print(f'problem with {i}')
        else:
            complete_sp.append(i)
    else:
        complete_sp.append(i)
    

In [413]:
len(incomplete_sp)

2

In [414]:
incomplete_sp

[115, 341]

In [415]:
len(complete_sp)

150

In [416]:
len(bad_sp)

0

In [369]:
bad_sp

[]

# Delete old single point calculation files

In [431]:
# for i in bad_sp:
for i in [115]:
    cf_files = glob.glob(os.path.join(working_dir, f'species_{i:04}', 'conformer.*'))
    for j in cf_files:
        os.remove(j)

# Remake single point files

In [432]:
# for i in bad_sp + incomplete_sp:
# for i in bad_sp:
for i in [115]:

    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    gaussian_log = os.path.join(sp_dir, 'sp.log')
    # make an orca file
    
    try:
        my_log = arkane.ess.ess_factory(gaussian_log)
    except arkane.exceptions.LogError:
        print(f'skipping bad gaussian file {i}')
        continue
    # make a run file
    coord, number, mass = my_log.load_geometry()
    my_atoms = ase.Atoms(number, coord)
    
    
    parallel = True
    orca_input_file = os.path.join(sp_dir, 'conformer.inp')
    input_format = """!{res}HF {level_of_theory} TightSCF tightPNO
!energy

%maxcore 35000
{opt_parallel_line}

* xyz {charge} {mult}
{xyz}*


"""
    charge = database.reference_sets['main'][i].charge
    multiplicity = database.reference_sets['main'][i].multiplicity
    input_content = input_format.format(
        res='r' if multiplicity == 1 else 'u',
        level_of_theory='dlpno-ccsd(t)-f12 cc-pvtz-f12 aug-cc-pvtz/c cc-pvtz-f12-cabs',
        opt_parallel_line='%pal nprocs 4 end' if parallel else '',
        charge=charge,
        mult=multiplicity,
        xyz=autotst_wrapper.get_xyz(my_atoms),
    )
    
    with open(orca_input_file, 'w') as f:
        f.write(input_content)
        
    run_orca_script = os.path.join(sp_dir, 'run_orca.sh')
    with open(run_orca_script, 'w') as f:
        # TODO format the text without """ so it doesn't mess with VSCode's collapse function button
        f.write("""#!/bin/bash
#SBATCH --job-name=""" + f'orca_{i:04}' + """
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem-per-cpu=50Gb
#SBATCH --time=24:00:00
#SBATCH --ntasks=4


ompi=/work/westgroup/orca/openmpi-4.1.6/build
PATH=$ompi/bin:$PATH
LD_LIBRARY_PATH=$ompi/lib:$ompi/etc:$LD_LIBRARY_PATH

#Orca
orcadir=/work/westgroup/orca/orca_6_0_1_linux_x86-64_shared_openmpi416
export PATH=$PATH:$orcadir
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$orcadir

$orcadir/orca conformer.inp > conformer.out
""")

In [35]:
my_calcs = sorted(bad_sp + incomplete_sp)

In [38]:
autotst_wrapper.ordered_array_str(my_calcs)

'95,127-127,184-184,225-225,246-247,261-262,265-265,282-282,292-292,296-297,303-303,306-306,329-329,331-331,335-335,339-339,343-343,353-354,373-373,376-376,384-384,389-390,392-392,396-396,399-400,402-403,406-406,411-413,415-416'

In [37]:
print(my_calcs)

[95, 127, 184, 225, 246, 247, 261, 262, 265, 282, 292, 296, 297, 303, 306, 329, 331, 335, 339, 343, 353, 354, 373, 376, 384, 389, 390, 392, 396, 399, 400, 402, 403, 406, 411, 412, 413, 415, 416]


In [417]:
total_incomplete = sorted(list(set(special_set) - set(complete_sp)))

In [418]:
total_incomplete

[115, 301, 341]

In [419]:
for i in total_incomplete:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    print(i, '\t', database_fun.get_unique_species_index(ref_sp), database.reference_sets['main'][i].smiles)

115 	 968 [CH2]C1=CC=CC=C1
301 	 1016 CC1CCCC1
341 	 1024 [O]C1=CC=CC=C1


In [420]:
inconsistent

[]

In [421]:
bad_geo

[301]

In [422]:
bad_sp

[]

## Define things for addition to database

In [394]:
method = 'M062X/ccpvtz'
freq_lot = arkane.modelchem.LevelOfTheory(method='M062X2023',
                           basis='ccpvtz',
                           software='gaussian',
                           )
energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023',
                           basis='ccpvtzf12',
                           software='orca',
                           )
freq_log = '/scratch/harris.se/guassian_scratch/bac/species_0010/sp.log'
energy_log = '/scratch/harris.se/guassian_scratch/bac/species_0010/conformer.out'
log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, energy_dict=None, temp=298.15)

periodic_table = rdkit.Chem.GetPeriodicTable()

def log_to_xyz_dict(log):
    log = arkane.ess.ess_factory(log)
    coords, nums, _ = log.load_geometry()
    syms = [arkane.common.symbol_by_number[int(n)] for n in nums]
    return {
        'coords': coords,
        'isotopes': [periodic_table.GetMostCommonIsotope(s) for s in syms],
        'symbols': syms
    }

level_of_theories = {
    'dlpno-ccsdt-f12-ccpvtz': arkane.modelchem.CompositeLevelOfTheory(freq=arkane.modelchem.LevelOfTheory(method='m062x', basis='ccpvtz', software='gaussian'),
                                                     energy=arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='cc-pvtz-f12', software='orca')),
    'M062X/ccpvtz': arkane.modelchem.LevelOfTheory(method='M062X',
                           basis='ccpvtz',
                           software='gaussian',
                           ),
    'dlpno-ccsd(t)-f12-2023': arkane.modelchem.LevelOfTheory(method='dlpnoccsd(t)f122023',
                           basis='ccpvtzf12',
                           software='orca',
                           ),
}

freq_scale_factors = {
    'M062X/ccpvtz': 0.955,
    'M062X/def2tzvp': 0.984,
    'dlpno-ccsdt-f12-ccpvtz': 1.002,
}
def log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=None, energy_dict=None, temp=298.15):

    freq_log = arkane.ess.ess_factory(freq_log)
    
    conformer, _ = freq_log.load_conformer()
    
    # Perform quick checks
    assert conformer.spin_multiplicity > 0
    assert any(isinstance(mode, rmgpy.statmech.IdealGasTranslation) for mode in conformer.modes)
    assert any(isinstance(mode, (rmgpy.statmech.LinearRotor, rmgpy.statmech.NonlinearRotor)) for mode in conformer.modes)
    assert any(isinstance(mode, rmgpy.statmech.HarmonicOscillator) for mode in conformer.modes)
    
    coords, nums, masses = freq_log.load_geometry()
    assert len(nums) > 1
    
    atoms = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in nums])
    conformer.coordinates = (coords, 'angstroms')
    conformer.number = nums
    conformer.mass = (masses, 'amu')
    
    freq_scale_factor = freq_scale_factors[method]
    frequencies = conformer.modes[2].frequencies.value_si
    for mode in conformer.modes:
        if isinstance(mode, rmgpy.statmech.HarmonicOscillator):
            mode.frequencies = (frequencies * freq_scale_factor, "cm^-1")
    if freq_scale_factor == 1:
        print('WARNING: Frequency scale factor is 1')
    zpe_scale_factor = freq_scale_factor / 1.014
    
    # get electronic energy (choose one option)
    # option 1: read from the log
    if energy_log is not None:
        energy_log = arkane.ess.ess_factory(energy_log)
        energy = energy_log.load_energy(zpe_scale_factor=zpe_scale_factor)  # J/mol

    # option 2: just read from a dictionary provided 
#     energy = energy_dict[int(label)] * 627.5094740631 * 4184
    
    # add ZPE
    energy += freq_log.load_zero_point_energy() * zpe_scale_factor if len(nums) > 1 else 0  # J/mol
    
    # add AECs
#     print(get_atom_correction(energy_lot, atoms) )
    energy += arkane.encorr.corr.get_atom_correction(energy_lot, atoms)  # J/mol
    conformer.E0 = (energy / 4184, 'kcal/mol')
    
    return rmgpy.quantity.ScalarQuantity((conformer.get_enthalpy(temp) + conformer.E0.value_si) / 4184, 'kcal/mol')


## Sanity Check on the calculations I've run

In [423]:
# make sure the orca geometry has the right atom counts

for i in complete_sp:
    orca_log = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    geo, nums, mass = arkane.ess.ess_factory(orca_log).load_geometry()
    atoms = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in nums])
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    ref_nums = [atom.number for atom in ref_sp.molecule[0].atoms]
    atoms2 = collections.Counter([arkane.common.symbol_by_number[int(n)] for n in ref_nums])
    if atoms != atoms2:
        print(i, 'difference!')

In [425]:
# look at any species where the H298 values vary by more than 5kcal/mol
for i in complete_sp:
    energy_log = f'/scratch/harris.se/guassian_scratch/bac/species_{i:04}/conformer.out'
    freq_log = f'/scratch/harris.se/guassian_scratch/bac/species_{i:04}/sp.log'
    if not (os.path.exists(energy_log) and os.path.exists(freq_log)):
        print(f'missing {i} files')
        continue
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    
    
    method = 'M062X/ccpvtz'
    freq_lot = arkane.modelchem.LevelOfTheory(method='M062X', basis='ccpvtz', software='gaussian')
    energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='ccpvtzf12', software='orca')

    hf298 = log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, temp=298.15)    
    
    
    # see if ref_db is off by too much
    my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]
    ref_h298 = database.reference_sets['main'][i].reference_data[my_key].thermo_data.H298
    kcal_diff = np.abs(hf298.value_si - ref_h298.value_si) / 4184
    if kcal_diff > 6.0:
        print(i, f'{kcal_diff:.3f} kcal\t\t{hf298.value_si / 4184:.3f}\t{ref_h298.value_si / 4184:.3f}')

335 6.815 kcal		40.690	33.875


# Prepare the database for new additions

In [103]:
ref_spcs = {spc.index: spc for spc in database.reference_sets['main']}

In [105]:
lot = arkane.modelchem.CompositeLevelOfTheory(
    freq=arkane.modelchem.LevelOfTheory(method='m062x',basis='ccpvtz',software='gaussian'),
    energy=arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023',basis='ccpvtzf12',software='orca')
)


In [141]:
database.reference_sets['main'][i].reference_data['ATcT'].thermo_data.H298.value_si

-11320.0

## Clean out previous attempts from the DB

In [433]:
already_in_db = []
for i in range(len(database.reference_sets['main'])):

    if lot in database.reference_sets['main'][i].calculated_data.keys():
        already_in_db.append(i)
        
print(f'Found out {len(already_in_db)} entries')
    

Found out 150 entries


In [429]:
clearing_out = []
for i in range(len(database.reference_sets['main'])):

    if lot in database.reference_sets['main'][i].calculated_data.keys():
        clearing_out.append(i)
        database.reference_sets['main'][i].calculated_data.pop(lot)
        
print(f'Cleared out {len(clearing_out)} entries')
    

Cleared out 0 entries


In [430]:
skip = []
added_to_database = []
for i in complete_sp:
    if i in skip:
        continue
    energy_log = f'/scratch/harris.se/guassian_scratch/bac/species_{i:04}/conformer.out'
    freq_log = f'/scratch/harris.se/guassian_scratch/bac/species_{i:04}/sp.log'
    if not (os.path.exists(energy_log) and os.path.exists(freq_log)):
        print(f'missing {i} files')
        continue
    
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    method = 'M062X/ccpvtz'
    freq_lot = arkane.modelchem.LevelOfTheory(method='M062X', basis='ccpvtz', software='gaussian')
    energy_lot = arkane.modelchem.LevelOfTheory(method='dlpno-ccsd(t)-f12-2023', basis='ccpvtzf12', software='orca')
    hf298 = log_to_enthalpy(method, freq_lot, energy_lot, freq_log, energy_log=energy_log, temp=298.15)    
    
    
    # see if ref_db is off by too much
    my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]
    ref_h298 = database.reference_sets['main'][i].reference_data[my_key].thermo_data.H298
    kcal_diff = np.abs(hf298.value_si - ref_h298.value_si) / 4184
    if kcal_diff > 6.0:
        print(i, f'{kcal_diff:.3f} kcal\t\t{hf298.value_si / 4184:.3f}\t{ref_h298.value_si / 4184:.3f}')
    

    # Update thermo
    try:
        database.reference_sets['main'][i].calculated_data[lot] = arkane.encorr.reference.CalculatedDataEntry(
            rmgpy.thermo.ThermoData(H298=hf298),
            xyz_dict=log_to_xyz_dict(energy_log)
        )
    except KeyError:
        print(f'Failed to add {i} to the database')
        continue
    added_to_database.append(i)
#     print(f'Added {i} to the database')
print(f'Added {len(added_to_database)} entries to database')

335 6.815 kcal		40.690	33.875
Added 150 entries to database


In [379]:
my_key = [key for key in database.reference_sets['main'][i].reference_data.keys()][0]

In [380]:
n = 335
database.reference_sets['main'][n].smiles

'[O-][O+]=O'

In [381]:
database.reference_sets['main'][n].reference_data[my_key].thermo_data.H298.value_si / 4184

33.87523900573614

In [382]:
keys = list(database.reference_sets['main'][n].calculated_data.keys())

In [384]:
ref_sp = rmgpy.species.Species(smiles=database.reference_sets['main'][n].smiles)

In [385]:
database_fun.get_unique_species_index(ref_sp)

1022

In [388]:
hf298

(40.6901,'kcal/mol')

In [389]:
for key in database.reference_sets['main'][n].calculated_data.keys():
    print(database.reference_sets['main'][n].calculated_data[key].thermo_data.H298)

37.3099 kcal/mol
37.3723 kcal/mol
36.8568 kcal/mol
42.4723 kcal/mol
29.9984 kcal/mol
34.0853 kcal/mol
27.304 kcal/mol
46.9234 kcal/mol
34.9602 kcal/mol
40.1101 kcal/mol
40.4086 kcal/mol
39.8387 kcal/mol
33.9388 kcal/mol
50.6587 kcal/mol
44.1321 kcal/mol
30.2886 kcal/mol
42.5993 kcal/mol
44.3255 kcal/mol
45.3405 kcal/mol
45.9869 kcal/mol


In [453]:
# key = keys[5]
# #     print(database.reference_sets['main'][n].calculated_data[key].xyz_dict)

# coords = database.reference_sets['main'][n].calculated_data[key].xyz_dict['coords']
# syms = database.reference_sets['main'][n].calculated_data[key].xyz_dict['symbols']
# nums = [periodic_table.GetAtomicNumber(sym) for sym in syms]
# atoms = ase.Atoms(nums, coords)


# ase.visualize.view(atoms, viewer='x3d')


In [203]:
print(autotst_wrapper.get_xyz(atoms))

C       0.706806000000000      0.225565000000000     -0.000000000000000
O      -0.545586000000000     -0.153630000000000      0.000000000000000
H       1.251055000000000     -0.746437000000000     -0.000000000000000
H      -1.114164000000000      0.632972000000000      0.000000000000000



# Actually run BAC Fitting

In [450]:
my_bac_job = arkane.encorr.bac.BACJob(
    lot,  # level of theory
    exclude_elements=['S', 'N', 'Cl', 'F']
)
my_bac_job.execute()

In [451]:
# my_bac_job.bac.fit(exclude_elements=['S', 'N', 'Cl', 'F'])

In [452]:
my_bac_job.write_output('.')